In [ ]:
from pathlib import Path

for f in Path("../data/raw/cornie").rglob("*"):
    print(f.name)

In [ ]:
import geopandas as gpd
import pandas as pd

In [ ]:
cornie = gpd.read_file(
    "../data/raw/cornie/cornie.shp"
)

cornie.head()

In [ ]:
print(cornie.shape)

cornie.columns.tolist()

In [ ]:
cornie.crs

In [ ]:
cornie.geom_type.value_counts()

In [ ]:
cornie["Code_18"].value_counts().sort_index()

In [ ]:

wind_gdf = gpd.read_parquet(
    "../data/interim/repd_processed/wind_farm_crs.parquet"
)

In [ ]:
wind_gdf.head()

In [ ]:
type(wind_gdf)

In [ ]:
wind_gdf.crs

In [ ]:
wind_cornie = gpd.sjoin(
    wind_gdf,
    cornie[["Code_18", "geometry"]],
    how="left",
    predicate="within"
)

In [ ]:
wind_cornie.head()

In [ ]:
wind_cornie["Code_18"].isna().sum()

In [ ]:
wind_cornie["Code_18"].value_counts().sort_index()

In [ ]:
wind_cornie[
    wind_cornie["Code_18"].isna()
][
    [
        "Site Name",
        "Technology Type",
        "Country"
    ]
]

In [ ]:
wind_cornie.groupby(
    "Technology Type"
)["Code_18"].apply(
    lambda x: x.isna().sum()
)

In [ ]:
print("Total farms:", len(wind_cornie))

print("Matched Farms:",
      wind_cornie["Code_18"].notna().sum())


print("Unmatched Farms:",
      wind_cornie["Code_18"].isna().sum())

In [ ]:
wind_cornie[
    (wind_cornie["Code_18"].isna()) &
    (wind_cornie["Technology Type"] == "Wind Onshore")
][
    [
        "Site Name",
        "X-coordinate",
        "Y-coordinate",
        "Region",
        "Country"
    ]
]

In [ ]:
unmatched = wind_cornie[
    wind_cornie["Code_18"].isna()
]

ax = cornie.plot(
    figsize= (10,10),
    alpha=0.4
)

unmatched.plot(
    ax=ax,
    color="red",
    markersize=50
)

In [ ]:
wind_cornie["landcover_status"] = "Matched"

wind_cornie.loc[
    wind_cornie["Technology Type"] == "Wind Offhosre",
    "landcover_status"
] = "Offshore"

wind_cornie.loc[
    wind_cornie["Code_18"].isna() &
    (wind_cornie["Technology Type"] == "Wind Onshore"),
    "landcover_status"
] = "Unclassified"

In [ ]:
wind_cornie.tail()

In [ ]:
wind_cornie.to_parquet(
    "../data/interim/cornie_processed/wind_cornie.parquet",
    index=False
)

In [ ]:
wind_cornie.to_csv(
    "../data/interim/cornie_processed/wind_cornie.csv",
    index=False
)

In [ ]:
wind_cornie.to_file(
    "../data/interim/cornie_processed/wind_cornie.shp"
)